# TD 5 : quand les lignes ont un ordre

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. lire une autocorrélation et en déduire le bon baseline
2. construire une table supervisée sans utiliser le futur
3. faire un backtest à plis glissants et lire son étendue
4. reconnaître un signal qui ne se prévoit pas, avant d'y passer un mois

## Prérequis

- avoir fini le TD 2
- avoir suivi le bloc E de la journée 1

**Durée :** 55 minutes

## Le contexte

Combien de passages aux urgences demain ? On prevoit le LENDEMAIN, chaque jour, et on evalue sur des fenetres de 7 jours consecutifs. 736 jours de releves quotidiens, **quatre départements bretons**, en open data. Pas la France entière : quatre départements, et le préciser change ce que l'on peut conclure.

## Le déroulé

| quand | quoi |
|---|---|
| 6:25 | étape 1 : la structure de la semaine |
| 6:40 | étape 2 : la table, sans utiliser le futur |
| 6:55 | étape 3 : le backtest, et son étendue |
| 7:10 | étape 4 : le signal qui ne se prévoit pas |
| 7:20 | TD 6 |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : La structure de la semaine

**15 minutes.** Avant tout modèle, regarder si le signal a une forme, et laquelle.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# charger et construire la serie nationale quotidienne
u = pd.read_parquet(DONNEES / 'urgences.parquet')
tous = u[u['age_label'] == 'tous ages']
serie = tous.groupby('date')['tot_pass_emgy'].sum().sort_index()

# autocorrelation a k jours
serie.autocorr(7)

# moyenne par jour de la semaine
serie.groupby(serie.index.dayofweek).mean()

# decaler une serie de k pas
serie.shift(7)

# moyenne glissante sur 7 pas
serie.shift(7).rolling(7).mean()

# erreur absolue moyenne
float(np.mean(np.abs(réel - prevu)))
```

### 1.1 Construire la série et lire sa mémoire

In [ ]:
# A FAIRE (1) : la serie quotidienne, toutes tranches d age confondues.
serie = ...

# A FAIRE (2) : les autocorrelations a 1, 7 et 364 jours.
print(f'{len(serie)} jours, moyenne {serie.mean():,.0f} passages par jour')
for k in (1, 7, 364):
    print(f'  autocorrelation a {k:>3} jours : {serie.autocorr(k):+.4f}')

**Indice 1.** Le premier patron du mémo. Attention : la tranche d'âge est écrite `'tous âges'`, avec l'accent.

**Indice 2.** `u = pd.read_parquet(DONNEES / 'urgences.parquet')`, puis filtrer sur `age_label`, puis `groupby('date')['tot_pass_emgy'].sum().sort_index()`

In [ ]:
assert len(serie) == 736, f'attendu 736 jours, obtenu {len(serie)}'
assert abs(serie.mean() - 2297) < 20, (
    f'moyenne attendue vers 2 297, obtenue {serie.mean():,.0f}. '
    'Avez-vous bien filtre sur la tranche tous ages ?')
a1, a7 = serie.autocorr(1), serie.autocorr(7)
assert a7 > a1, 'la memoire a 7 jours doit depasser celle a 1 jour'
print(f'correct. a 1 jour {a1:+.4f}, a 7 jours {a7:+.4f}.')
print()
print('Hier ressemble mal a aujourd hui. La semaine dernière, beaucoup mieux.')
print('Votre baseline sera donc la valeur d il y a 7 jours, pas celle d hier.')
print('Cette decision est prise AVANT d ecrire la moindre ligne de modele.')

### 1.2 Pourquoi la semaine ?

In [ ]:
# A FAIRE : la moyenne de passages par jour de la semaine.
par_jour = ...

noms = ['lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi', 'samedi', 'dimanche']
for i, nom in enumerate(noms):
    print(f'  {nom:<10} {par_jour[i]:>8,.0f}')

**Indice 1.** Le patron `groupby(serie.index.dayofweek).mean()` du mémo. Lundi vaut 0.

**Indice 2.** `par_jour = serie.groupby(serie.index.dayofweek).mean()`

In [ ]:
assert par_jour.idxmax() == 0, (
    f'le pic devrait etre le lundi (0), vous trouvez {par_jour.idxmax()}')
ecart = par_jour[0] / par_jour[5] - 1
print(f'Le lundi depasse le samedi de {par_jour[0] - par_jour[5]:,.0f} '
      f'passages, soit {ecart:+.1%}.')
print()
print('Ce n est pas un artefact : c est l accumulation du week-end qui se')
print('presente a la reouverture des cabinets de ville. Une structure que')
print('l on peut expliquer est une structure sur laquelle on peut compter.')

---

## Étape 2 : La table, sans utiliser le futur

**15 minutes.** Une prévision à 7 jours interdit d'utiliser la valeur d'hier.

**Ce que l'on prévoit exactement, et c'est la décision qui commande tout le reste.** Ici : **le lendemain**. Chaque matin, le service veut le chiffre du jour suivant, et il connaît celui de la veille.

Donc `lag_1` est **légitime**, parce qu'au moment de produire la prévision de demain, la valeur d'hier existe bel et bien.

> **Changez l'horizon et tout change.** Si l'on prévoyait à 7 jours, `lag_1` deviendrait une fuite : au moment de la prévision, la valeur de la veille du jour visé n'aurait pas encore eu lieu. L'exercice bonus de fin de notebook vous fera mesurer ce que cette seule différence coûte.

In [ ]:
# A FAIRE : completer la table. On prevoit le lendemain, donc le decalage
#           minimum est de 1 jour.
def table_retard(s, retard_min=1):
    t = pd.DataFrame({'y': s})
    for r in [retard_min, retard_min + 1, retard_min + 2, 7, 14]:
        t[f'lag_{r}'] = ...
    t['moy7'] = ...           # moyenne des 7 jours precedant le decalage
    t['moy28'] = ...
    t['jour'] = s.index.dayofweek
    t['mois'] = s.index.month
    return t.dropna()

tbl = table_retard(serie)
feats = [c for c in tbl.columns if c != 'y']
X, y = tbl[feats].values, tbl['y'].values

print(f'{len(tbl)} lignes x {len(feats)} features')
print(feats)

**Indice 1.** `s.shift(r)` décale de r pas. Pour les moyennes glissantes, décalez d'abord puis `rolling(...).mean()`, dans cet ordre.

**Indice 2.** `t[f'lag_{r}'] = s.shift(r)`, `t['moy7'] = s.shift(retard_min).rolling(7).mean()`

In [ ]:
assert 'lag_1' in feats, (
    'a un horizon de 1 jour, lag_1 est legitime : la veille est connue')
assert 'moy7' in feats and 'moy28' in feats, 'les moyennes glissantes manquent'
assert len(tbl) > 650, f'seulement {len(tbl)} lignes apres dropna'
# la table a ete dropna, donc on repere la ligne par sa DATE et non sa position
jour = tbl.index[40]
pos = serie.index.get_loc(jour)
ok, attendu = tbl['moy7'].iloc[40], serie.iloc[pos - 7:pos].mean()
assert abs(ok - attendu) < 1e-6, (
    'moy7 doit etre la moyenne des 7 jours PRECEDANT le decalage. '
    'Decalez d abord, puis rolling : l ordre change ce que la feature contient.')
print(f'correct : {len(tbl)} lignes, {len(feats)} features.')
print('Chaque feature existera au moment de produire la prevision du lendemain.')

---

## Étape 3 : Le backtest, et son étendue

**15 minutes.** On coupe dans le temps, plusieurs fois, et on regarde autant la dispersion que la moyenne.

In [ ]:
# A FAIRE : 12 plis, fenetre de test de 7 jours, decalee de 28 jours.
#           Chaque pli s entraine sur TOUT ce qui precede sa fenetre de test.
def plis(n, n_folds=12, fenetre=7, pas=28):
    debut_min = n - ((n_folds - 1) * pas + fenetre)
    for k in range(n_folds):
        fin = debut_min + k * pas
        yield ...        # (train_idx, test_idx)

decoupes = list(plis(len(y)))
print(f'{len(decoupes)} plis')
print(f'premier entrainement : {len(decoupes[0][0])} jours')
print(f'dernier entrainement : {len(decoupes[-1][0])} jours')

**Indice 1.** Le train va de 0 à `fin`, le test de `fin` à `fin + fenetre`. La fenêtre fait 7 jours, l'horizon reste de 1 jour. `np.arange` construit les deux.

**Indice 2.** `yield np.arange(0, fin), np.arange(fin, fin + fenetre)`

In [ ]:
assert len(decoupes) == 12, f'attendu 12 plis, obtenu {len(decoupes)}'
for tr, te in decoupes:
    assert tr.max() < te.min(), (
        'un pli s entraine sur des jours posterieurs a sa fenetre de test')
    assert len(te) == 7, 'chaque pli doit tester sur 7 jours'
print('correct : 12 plis, aucun ne s entraine sur son propre futur.')
print('La fenetre d entrainement grandit a chaque pli, comme en production.')

### 3.1 Les baselines, puis le modèle

In [ ]:
# A FAIRE : scorer trois baselines et un modele sur les 12 plis.
from sklearn.ensemble import HistGradientBoostingRegressor

def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

scores = {'naif (hier)': [], 'naif (il y a 7 jours)': [],
          'moyenne des 28 derniers': [], 'gradient boosting': []}
for tr, te in decoupes:
    scores['naif (hier)'].append(...)
    scores['naif (il y a 7 jours)'].append(...)
    scores['moyenne des 28 derniers'].append(...)
    modele = ...
    scores['gradient boosting'].append(...)

print(f"{'methode':<28}{'MAE':>9}{'min':>9}{'max':>9}")
moyennes = {}
for nom, vals in scores.items():
    moyennes[nom] = float(np.mean(vals))
    print(f'{nom:<28}{np.mean(vals):>9.1f}{min(vals):>9.1f}{max(vals):>9.1f}')

**Indice 1.** Répéter hier, c'est `y[te - 1]`. Répéter la semaine dernière, `y[te - 7]`. La moyenne des 28 derniers est `np.repeat(y[tr][-28:].mean(), len(te))`.

**Indice 2.** `modele = HistGradientBoostingRegressor(random_state=42).fit(X[tr], y[tr])` puis `mae(y[te], modele.predict(X[te]))`

In [ ]:
gbm = moyennes['gradient boosting']
meilleur_naif = min(moyennes['naif (hier)'],
                    moyennes['naif (il y a 7 jours)'],
                    moyennes['moyenne des 28 derniers'])
assert gbm < meilleur_naif, 'le modele doit battre le meilleur naif ici'
assert moyennes['naif (il y a 7 jours)'] < moyennes['naif (hier)'], (
    'repeter la semaine dernière doit battre repeter hier, '
    'c est ce que l autocorrelation annoncait')
vals = scores['gradient boosting']
etendue = max(vals) / min(vals)
print(f'le modele fait {1 - gbm / meilleur_naif:+.1%} par rapport au '
      'meilleur naif.')
print()
print('Notez au passage que repeter la semaine dernière bat largement')
print('repeter hier. L autocorrelation de l etape 1 l avait annonce.')
print()
print(f'Et maintenant l etendue : de {min(vals):.1f} a {max(vals):.1f},')
print(f'soit un facteur {etendue:.1f} entre la meilleure et la pire semaine.')
print()
print('Annoncer la moyenne seule promet une regularite que le modele n a pas.')
print(f'La phrase honnete est : environ {min(vals):.0f} de biais une semaine')
print(f'ordinaire, environ {max(vals):.0f} une semaine inhabituelle, et nous')
print('ne savons pas distinguer les deux a l avance.')

---

## Étape 4 : Le signal qui ne se prévoit pas

**10 minutes.** Le même code, un autre indicateur du même fichier. C'est l'étape à ne pas manquer.

**Pariez avant de lancer.** On va reprendre exactement le même pipeline, les mêmes dates, les mêmes départements, les mêmes features, les mêmes plis et la même graine. On change uniquement **la colonne à prévoir**.

`nb_pass_emgy_covid` chez les 5-14 ans : un indicateur étroit, sur le même fichier.

In [ ]:
# A FAIRE : le meme pipeline sur le second indicateur.
jeunes = u[u['age_label'] == '5-14 ans']
contraste = ...

for nom, s in (('passages totaux, tous ages', serie),
               ('passages covid, 5-14 ans', contraste)):
    t = table_retard(s)
    f = [c for c in t.columns if c != 'y']
    Xc, yc = t[f].values, t['y'].values
    dec = list(plis(len(yc)))
    naifs, modeles = [], []
    for tr, te in dec:
        naifs.append(mae(yc[te], yc[te - 7]))
        m = HistGradientBoostingRegressor(random_state=42).fit(Xc[tr], yc[tr])
        modeles.append(mae(yc[te], m.predict(Xc[te])))
    n_moy, m_moy = float(np.mean(naifs)), float(np.mean(modeles))
    print(f'{nom:<30} autocorr 7j {s.autocorr(7):+.4f}   '
          f'naif {n_moy:>7.2f}   modele {m_moy:>7.2f}   '
          f'gain {1 - m_moy / n_moy:+.1%}')

**Indice 1.** Même construction que `serie`, mais sur la colonne `nb_pass_emgy_covid` et la tranche `'5-14 ans'`.

**Indice 2.** `contraste = jeunes.groupby('date')['nb_pass_emgy_covid'].sum().sort_index().astype(float)`

In [ ]:
t2 = table_retard(contraste)
f2 = [c for c in t2.columns if c != 'y']
X2, y2 = t2[f2].values, t2['y'].values
d2 = list(plis(len(y2)))
n2 = float(np.mean([mae(y2[te], y2[te - 7]) for tr, te in d2]))
m2 = float(np.mean([
    mae(y2[te], HistGradientBoostingRegressor(random_state=42)
        .fit(X2[tr], y2[tr]).predict(X2[te])) for tr, te in d2]))
assert m2 > n2, (
    'sur ce second indicateur, le modele devrait etre BATTU par le naif')
print(f'Sur le second indicateur, le modele est {m2 / n2 - 1:.1%} PIRE')
print('que de repeter simplement la valeur de la semaine dernière.')
print()
print('Rien n a change que le signal. Memes dates, memes departements,')
print('memes features, memes plis, meme graine, meme code.')
print()
print(f'Et l autocorrelation a 7 jours le disait : {serie.autocorr(7):+.4f}')
print(f'contre {contraste.autocorr(7):+.4f}, AVANT d ecrire une ligne de modele.')

## Ce que cette étape a établi

- L'autocorrélation choisit le baseline avant tout modèle : +0,59 à 7 jours contre +0,25 à 1 jour.
- Le pic du lundi est explicable, donc fiable : l'accumulation du week-end.
- À 7 jours d'horizon, aucun lag inférieur à 7. `lag_1` est une fuite temporelle.
- Le modèle gagne nettement sur le signal structuré, et la moyenne cache un facteur proche de 5 entre les plis.
- Sur un autre indicateur du même fichier, le modèle perd contre le naïf. La prévisibilité appartient au signal.

## Les erreurs classiques

- **Oublier de trier la série par date.** Les lags pointent alors n'importe où, et la MAE explose sans explication.
- **Utiliser `lag_1` pour un horizon de 7 jours.** C'est une fuite : cette valeur n'existera pas au moment de la prévision.
- **Annoncer la MAE moyenne seule.** Donnez l'étendue entre plis, sinon vous promettez une régularité absente.

## Pour aller plus loin, si vous avez fini en avance

- **L'exercice bonus qui compte.** Refaire l'étape 3 avec `retard_min=7` au lieu de 1, c'est à dire en se plaçant à un horizon de 7 jours. Tout le reste identique. Le modèle bat-il encore le naïf ? Et que dit ce résultat sur la valeur réelle de `lag_1` ?
- Chercher dans le fichier un troisième indicateur et prédire son comportement à partir de sa seule autocorrélation, avant de le mesurer.